# ML-QEM: LiH (6-Qubit) Data Generation

Generates (noisy, ideal) Pauli expectation value pairs for the 6-qubit LiH molecule.

**Key differences from H2 (2 qubits, 5 Pauli terms):**
- 6 qubits, 62 Pauli terms
- Active space: (2 electrons, 3 spatial orbitals) with Jordan-Wigner mapping
- Backend: FakeJakartaV2 (7 qubits)
- TwoLocal ansatz: 24 parameters, 15 CX gates, 48 SX gates

**2000 samples to match H2 dataset. Expected runtime: ~3 hours.**

In [1]:
import numpy as np
import json

from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit import transpile

from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import Estimator as AerEstimatorV1
from qiskit_ibm_runtime.fake_provider import FakeJakartaV2

print('All imports OK')

All imports OK


## 1. LiH Hamiltonian (6 qubits, 62 terms)

LiH at equilibrium bond length (1.595 A), STO-3G basis, active space (2e, 3o),
Jordan-Wigner mapping. Pre-computed with PySCF + qiskit-nature.

In [2]:
LIH_LABELS = [
    'IIIIII', 'IIIIIZ', 'IIIIYY', 'IIIIXX', 'IIIIZI', 'IIIZII', 'IIZIII',
    'IYYIII', 'IXXIII', 'IZIIII', 'ZIIIII', 'IIIIZZ', 'IIIZIZ', 'IIZIIZ',
    'IYYIIZ', 'IXXIIZ', 'IZIIIZ', 'ZIIIIZ', 'IIIZYY', 'IIIZXX', 'IIZIYY',
    'IIZIXX', 'IYYIYY', 'IXXIYY', 'IYYIXX', 'IXXIXX', 'IZIIYY', 'IZIIXX',
    'ZIIIYY', 'ZIIIXX', 'YZYYZY', 'XZXYZY', 'YZYXZX', 'XZXXZX', 'YYIYZY',
    'XXIYZY', 'YYIXZX', 'XXIXZX', 'IIIZZI', 'IIZIZI', 'IYYIZI', 'IXXIZI',
    'IZIIZI', 'ZIIIZI', 'YZYYYI', 'XZXYYI', 'YZYXXI', 'XZXXXI', 'YYIYYI',
    'XXIYYI', 'YYIXXI', 'XXIXXI', 'IIZZII', 'IYYZII', 'IXXZII', 'IZIZII',
    'ZIIZII', 'IZZIII', 'ZIZIII', 'ZYYIII', 'ZXXIII', 'ZZIIII'
]

LIH_COEFFS = [
    -0.4610943472, 0.0267929798, 0.0120165932, 0.0120165932, -0.1456736247,
    -0.1615415094, 0.0267929798, 0.0120165932, 0.0120165932, -0.1456736247,
    -0.1615415094, 0.052684777, 0.061742011, 0.1219144565, 0.0121237381,
    0.0121237381, 0.0559382693, 0.0676045404, 0.0033899272, 0.0033899272,
    0.0121237381, 0.0121237381, 0.0032534924, 0.0032534924, 0.0032534924,
    0.0032534924, -0.0018545502, -0.0018545502, -0.001428226, -0.001428226,
    0.0058625294, 0.0058625294, 0.0058625294, 0.0058625294, 0.0048181532,
    0.0048181532, 0.0048181532, 0.0048181532, 0.0601814949, 0.0559382693,
    -0.0018545502, -0.0018545502, 0.0844837494, 0.0705009434, 0.0048181532,
    0.0048181532, 0.0048181532, 0.0048181532, 0.0103194485, 0.0103194485,
    0.0103194485, 0.0103194485, 0.0676045404, -0.001428226, -0.001428226,
    0.0705009434, 0.0782363778, 0.052684777, 0.061742011, 0.0033899272,
    0.0033899272, 0.0601814949
]

pauli_ops = [SparsePauliOp(label) for label in LIH_LABELS]

full_op = SparsePauliOp.from_list(list(zip(LIH_LABELS, LIH_COEFFS)))
mat = full_op.to_matrix()
E_EXACT_LIH = float(np.min(np.linalg.eigvalsh(mat)))

print(f'LiH Hamiltonian: {len(LIH_LABELS)} Pauli terms on 6 qubits')
print(f'Exact ground state energy: {E_EXACT_LIH:.8f} Ha')
print(f'Bond length: 1.595 Angstrom (equilibrium)')

LiH Hamiltonian: 62 Pauli terms on 6 qubits
Exact ground state energy: -1.06010420 Ha
Bond length: 1.595 Angstrom (equilibrium)


## 2. Ansatz, Noise Model, Estimators

In [3]:
ansatz = TwoLocal(num_qubits=6, rotation_blocks='ry',
                  entanglement_blocks='cx', entanglement='linear', reps=3)
print(f'Ansatz parameters: {ansatz.num_parameters}')

transpiled_ansatz = transpile(ansatz, basis_gates=['rz', 'sx', 'cx', 'x'],
                              optimization_level=1)
ops = transpiled_ansatz.count_ops()
N_2Q = ops.get('cx', 0)
N_SX = ops.get('sx', 0)
print(f'Transpiled: depth={transpiled_ansatz.depth()}, '
      f'CX={N_2Q}, SX={N_SX}, qubits={transpiled_ansatz.num_qubits}')

backend = FakeJakartaV2()
noise_model = NoiseModel.from_backend(backend)
print(f'Backend: {backend.name} ({backend.num_qubits} qubits)')

ideal_estimator = StatevectorEstimator()
noisy_estimator = AerEstimatorV1()
noisy_estimator.set_options(noise_model=noise_model, shots=10000)
print('Estimators ready.')

Ansatz parameters: 24
Transpiled: depth=25, CX=15, SX=48, qubits=6
Backend: fake_jakarta (7 qubits)
Estimators ready.


## 3. Generate Training Data

**2000 random theta values x 62 observables = 124,000 evaluations per estimator.**

Matches H2 dataset size. Expected runtime: ~3 hours.

In [4]:
import time

N_SAMPLES  = 2000
N_OBS      = len(LIH_LABELS)
N_PARAMS   = ansatz.num_parameters

rng = np.random.default_rng(42)
theta_samples = rng.uniform(-np.pi, np.pi, size=(N_SAMPLES, N_PARAMS))

ideal_data = np.zeros((N_SAMPLES, N_OBS))
noisy_data = np.zeros((N_SAMPLES, N_OBS))

t0 = time.time()

for i, theta in enumerate(theta_samples):
    bound = transpiled_ansatz.assign_parameters(theta)

    # Ideal: StatevectorEstimator V2 API
    ideal_job = ideal_estimator.run([(bound, op) for op in pauli_ops])
    ideal_res = ideal_job.result()

    # Noisy: AerEstimatorV1 API
    noisy_job = noisy_estimator.run([bound] * N_OBS, pauli_ops)
    noisy_res = noisy_job.result()

    for j in range(N_OBS):
        ideal_data[i, j] = ideal_res[j].data.evs
        noisy_data[i, j] = noisy_res.values[j]

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (N_SAMPLES - i - 1)
        print(f'  {i+1}/{N_SAMPLES} done  '
              f'({elapsed/60:.1f} min elapsed, ~{eta/60:.0f} min remaining)')

total = time.time() - t0
print(f'\nDone in {total/60:.1f} minutes.')

  100/2000 done  (4.4 min elapsed, ~84 min remaining)
  200/2000 done  (9.1 min elapsed, ~82 min remaining)
  300/2000 done  (13.8 min elapsed, ~78 min remaining)
  400/2000 done  (18.1 min elapsed, ~72 min remaining)
  500/2000 done  (22.3 min elapsed, ~67 min remaining)
  600/2000 done  (26.5 min elapsed, ~62 min remaining)
  700/2000 done  (30.7 min elapsed, ~57 min remaining)
  800/2000 done  (34.8 min elapsed, ~52 min remaining)
  900/2000 done  (38.9 min elapsed, ~48 min remaining)
  1000/2000 done  (43.1 min elapsed, ~43 min remaining)
  1100/2000 done  (47.5 min elapsed, ~39 min remaining)
  1200/2000 done  (51.8 min elapsed, ~35 min remaining)
  1300/2000 done  (55.9 min elapsed, ~30 min remaining)
  1400/2000 done  (60.1 min elapsed, ~26 min remaining)
  1500/2000 done  (64.2 min elapsed, ~21 min remaining)
  1600/2000 done  (68.9 min elapsed, ~17 min remaining)
  1700/2000 done  (76.7 min elapsed, ~14 min remaining)
  1800/2000 done  (84.6 min elapsed, ~9 min remaining)
  19

## 4. Save Dataset

In [5]:
np.save('lih_ideal_data.npy', ideal_data)
np.save('lih_noisy_data.npy', noisy_data)
np.save('lih_theta_samples.npy', theta_samples)

meta = {
    'molecule':           'LiH',
    'bond_length':        1.595,
    'basis':              'sto-3g',
    'active_space':       '(2e, 3o)',
    'n_qubits':           6,
    'pauli_labels':       LIH_LABELS,
    'hamiltonian_coeffs': LIH_COEFFS,
    'exact_gs_energy':    E_EXACT_LIH,
    'n_samples':          N_SAMPLES,
    'n_params':           N_PARAMS,
    'shots':              10000,
    'noise_backend':      'FakeJakartaV2',
    'n_2q_gates':         N_2Q,
    'n_sx_gates':         N_SX,
}
with open('lih_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
print(f'  lih_ideal_data.npy  -- shape {ideal_data.shape}')
print(f'  lih_noisy_data.npy  -- shape {noisy_data.shape}')
print(f'  lih_theta_samples.npy -- shape {theta_samples.shape}')
print(f'  lih_meta.json')

Saved:
  lih_ideal_data.npy  -- shape (2000, 62)
  lih_noisy_data.npy  -- shape (2000, 62)
  lih_theta_samples.npy -- shape (2000, 24)
  lih_meta.json


## 5. Sanity Check

In [7]:
errors = np.abs(ideal_data - noisy_data)

print(f'{"Pauli":<10} {"Ideal mean":>12} {"Noisy mean":>12} {"MAE":>10}')
print('-' * 48)
for j in [0, 1, 4, 5, 11, 12, 13, 42, 61]:
    label = LIH_LABELS[j]
    print(f'{label:<10} {ideal_data[:, j].mean():>12.4f} '
          f'{noisy_data[:, j].mean():>12.4f} '
          f'{errors[:, j].mean():>10.4f}')


overall_mae = errors[:, 1:].mean()
print(f'\nOverall MAE (non-identity): {overall_mae:.4f}')
print(f'{"Good" if 0.005 < overall_mae < 0.3 else "WARNING"}: '
      f'noise level looks {"reasonable" if 0.005 < overall_mae < 0.3 else "unexpected"}.')

Pauli        Ideal mean   Noisy mean        MAE
------------------------------------------------
IIIIII           1.0000       1.0000     0.0000
IIIIIZ           0.0203       0.0193     0.0301
IIIIZI           0.0088       0.0080     0.0280
IIIZII          -0.0026      -0.0025     0.0189
IIIIZZ          -0.0040      -0.0035     0.0382
IIIZIZ           0.0011       0.0005     0.0247
IIZIIZ          -0.0048      -0.0043     0.0232
IZIIZI          -0.0007      -0.0003     0.0341
ZZIIII           0.0022       0.0018     0.0602

Overall MAE (non-identity): 0.0408
Good: noise level looks reasonable.
